In [0]:
# Databricks notebook source
# MAGIC %mdcatalog_name
# MAGIC # Bronze ingestion
# MAGIC
# MAGIC **Doel:** gegenereerde CSV-bronbestanden inlezen en opslaan als Delta-tabellen.
# MAGIC
# MAGIC **Input:** CSV-bestanden uit `data/generated`
# MAGIC
# MAGIC **Output:** Bronze-tabellen in `workspace.retail`

In [0]:
from pathlib import Path
import shutil

from pyspark.sql import functions as F

In [0]:
current_directory = Path.cwd()

print(f"Current directory: {current_directory}")

In [0]:
project_root = current_directory.parent
source_directory = project_root / "data" / "generated"


print(f"Project root: {project_root}")
print(f"Source directory: {source_directory}")

In [0]:
catalog_name = "workspace"
schema_name = "retail"
volume_name = "raw_files"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
COMMENT 'Retail marketing lakehouse project'
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}
COMMENT 'Raw source files used for Bronze ingestion'
""")

print(f"Schema ready: {catalog_name}.{schema_name}")
print(f"Volume ready: {catalog_name}.{schema_name}.{volume_name}")

In [0]:
spark.sql(f"SHOW VOLUMES IN {catalog_name}.{schema_name}").show(truncate=False)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
)

processed_files_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("processed_at", TimestampType(), False),
])

In [0]:
processed_files_table = f"{catalog_name}.{schema_name}.processed_order_files"

(
    spark.createDataFrame(
        [],
        processed_files_schema,
    )
    .write
    .format("delta")
    .mode("ignore")
    .saveAsTable(processed_files_table)
)

In [0]:
processed_order_files_df = spark.table(processed_files_table)

processed_order_files_df.printSchema()
processed_order_files_df.show()

In [0]:
# from pyspark.sql import Row
# from pyspark.sql import functions as F

# test_processed_files = [
#     Row(file_name="orders_001.csv"),
#     Row(file_name="orders_002.csv"),
#     Row(file_name="orders_003.csv"),
#     Row(file_name="orders_004.csv"),
# ]

# (
#     spark.createDataFrame(test_processed_files)
#     .withColumn("processed_at", F.current_timestamp())
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(processed_files_table)
# )

# print("Test data written.")

In [0]:
# spark.table(processed_files_table).show(truncate=False)